In [2]:
import pandas as pd

# ---------------------------------------------------------
# SETUP: Criação do Conjunto de Dados "Lentes de Contato"
# ---------------------------------------------------------
data = {
    'Idade': ['infantil', 'infantil', 'infantil', 'infantil', 'adolescente', 'adolescente', 'adolescente', 'adolescente', 'adolescente', 'adulto', 'adulto', 'adulto', 'adulto', 'adulto', 'adulto'],
    'Diagnóstico': ['miopia', 'miopia', 'hipermetropia', 'hipermetropia', 'miopia', 'miopia', 'miopia', 'hipermetropia', 'hipermetropia', 'miopia', 'miopia', 'miopia', 'hipermetropia', 'hipermetropia', 'hipermetropia'],
    'Astigmatismo': ['não', 'sim', 'não', 'sim', 'não', 'sim', 'não', 'não', 'sim', 'não', 'sim', 'sim', 'não', 'sim', 'não'],
    'Taxa_lacrimal': ['reduzida', 'normal', 'normal', 'normal', 'reduzida', 'reduzida', 'normal', 'reduzida', 'normal', 'normal', 'normal', 'normal', 'reduzida', 'normal', 'normal'],
    'Tipo_de_lente': ['nenhuma', 'gelatinosa', 'gelatinosa', 'dura', 'gelatinosa', 'nenhuma', 'dura', 'gelatinosa', 'dura', 'gelatinosa', 'dura', 'gelatinosa', 'nenhuma', 'gelatinosa', 'gelatinosa']
}

df = pd.DataFrame(data)

# Variáveis globais
alpha = 1
classes = df['Tipo_de_lente'].unique()
atributos = ['Idade', 'Diagnóstico', 'Astigmatismo', 'Taxa_lacrimal']
total_instancias = len(df)

# Função para calcular a probabilidade com Laplace
def prob_laplace(atributo, valor, classe_alvo):
    df_classe = df[df['Tipo_de_lente'] == classe_alvo]
    n_j = len(df_classe) # Total de instâncias da classe
    n_ij = len(df_classe[df_classe[atributo] == valor]) # Contagem do valor na classe
    v_i = df[atributo].nunique() # Quantidade de valores únicos do atributo

    prob = (n_ij + alpha) / (n_j + v_i)
    return prob, n_ij, n_j, v_i

# Gerando e imprimindo as tabelas para o Exercício 1
for atributo in atributos:
    valores_unicos = df[atributo].unique()
    print(f"--- Atributo: {atributo.upper()} ---")

    for valor in valores_unicos:
        linha = f"{valor:15} | "
        for c in classes:
            prob, n_ij, n_j, v_i = prob_laplace(atributo, valor, c)
            # Mostra a fração exata (ex: 2/6) e o valor decimal
            linha += f"P({c[:4]}): ({n_ij}+{alpha})/({n_j}+{v_i}) = {prob:.4f} | "
        print(linha)
    print()

# ---------------------------------------------------------
# EXERCÍCIO 2: Previsão Naive Bayes para o Paciente
# ---------------------------------------------------------

# Dados do paciente a ser classificado
paciente = {
    'Idade': 'infantil',
    'Diagnóstico': 'hipermetropia',
    'Astigmatismo': 'não',
    'Taxa_lacrimal': 'reduzida'
}

print(f"Paciente: {paciente}\n")

probabilidades_nao_normalizadas = {}

# 1. Calcular a probabilidade (peso) para cada classe
for c in classes:
    # Probabilidade a priori da classe P(C)
    p_classe = len(df[df['Tipo_de_lente'] == c]) / total_instancias

    # Multiplicar pelas probabilidades condicionais de cada atributo P(Xi | C)
    p_condicional_total = 1.0
    for atributo, valor in paciente.items():
        prob_laplace_val, _, _, _ = prob_laplace(atributo, valor, c)
        p_condicional_total *= prob_laplace_val

    prob_final = p_classe * p_condicional_total
    probabilidades_nao_normalizadas[c] = prob_final
    print(f"Valor (não normalizado) para a classe '{c}': {prob_final:.6f}")

# 2. Normalizar as probabilidades
soma_total = sum(probabilidades_nao_normalizadas.values())
print(f"\nSoma total dos valores: {soma_total:.6f}\n")

print("--- Probabilidades Normalizadas ---")
classe_prevista = None
maior_prob = -1

for c, valor in probabilidades_nao_normalizadas.items():
    prob_norm = valor / soma_total
    print(f"P({c} | Paciente) = {prob_norm * 100:.1f}%")

    if prob_norm > maior_prob:
        maior_prob = prob_norm
        classe_prevista = c

print(f"\n=> PREVISÃO FINAL DO CLASSIFICADOR: {classe_prevista.upper()}")

--- Atributo: IDADE ---
infantil        | P(nenh): (1+1)/(3+3) = 0.3333 | P(gela): (2+1)/(8+3) = 0.2727 | P(dura): (1+1)/(4+3) = 0.2857 | 
adolescente     | P(nenh): (1+1)/(3+3) = 0.3333 | P(gela): (2+1)/(8+3) = 0.2727 | P(dura): (2+1)/(4+3) = 0.4286 | 
adulto          | P(nenh): (1+1)/(3+3) = 0.3333 | P(gela): (4+1)/(8+3) = 0.4545 | P(dura): (1+1)/(4+3) = 0.2857 | 

--- Atributo: DIAGNÓSTICO ---
miopia          | P(nenh): (2+1)/(3+2) = 0.6000 | P(gela): (4+1)/(8+2) = 0.5000 | P(dura): (2+1)/(4+2) = 0.5000 | 
hipermetropia   | P(nenh): (1+1)/(3+2) = 0.4000 | P(gela): (4+1)/(8+2) = 0.5000 | P(dura): (2+1)/(4+2) = 0.5000 | 

--- Atributo: ASTIGMATISMO ---
não             | P(nenh): (2+1)/(3+2) = 0.6000 | P(gela): (5+1)/(8+2) = 0.6000 | P(dura): (1+1)/(4+2) = 0.3333 | 
sim             | P(nenh): (1+1)/(3+2) = 0.4000 | P(gela): (3+1)/(8+2) = 0.4000 | P(dura): (3+1)/(4+2) = 0.6667 | 

--- Atributo: TAXA_LACRIMAL ---
reduzida        | P(nenh): (3+1)/(3+2) = 0.8000 | P(gela): (2+1)/(8+2) = 0.